# Backtest Demo — XAUUSD M5 Strategy

End-to-end walkthrough of the backtest framework: load historical
OHLC, run the live strategy pipeline over it, and inspect the results
both as raw metrics and as charts.

**What this notebook does:**

1. Sets up the environment (Colab and local both supported)
2. Loads OHLC data — synthetic by default; swap in Dukascopy CSV for real runs
3. Configures the backtest knobs
4. Runs the engine and prints headline metrics
5. Renders the six charts inline
6. Writes a self-contained HTML report

**Important:** the strategy modules used here (`bot.strategy.*`) are the
**same modules** the live bot uses. A zone the backtest detects is the
zone the live bot would detect. This is the one guarantee that makes
backtest-driven tuning meaningful.

> The synthetic dataset in Section 2 may produce few or zero trades
> — the patterns are seeded but pass through the same strict
> Strong-Point + Imbalance filters as live data. **For meaningful
> results, replace it with a 1+ year Dukascopy CSV in Section 2.**


## Section 1 — Setup

Detect the runtime (Colab vs local), install missing deps, clone the repo,
and import everything the rest of the notebook needs.


In [ ]:
# Detect Colab vs local. Colab-specific calls (file upload/download)
# are gated on IN_COLAB so the notebook also runs in plain Jupyter.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    print("Running in Google Colab.")
except ImportError:
    IN_COLAB = False
    print("Running locally (not Colab).")


In [ ]:
# Install dependencies. Colab ships with pandas + numpy; plotly is
# usually present too but we ensure it's available. Loguru is small.
%pip install --quiet 'pandas>=2.2' 'numpy>=1.26,<2.0' 'plotly>=5.20' 'loguru>=0.7' 'pydantic>=2.5'


In [ ]:
# Clone (or pull) the bot repo so we can import bot.backtest.
# Local users with the repo already cloned can skip this cell.
import os, subprocess

REPO_URL = "https://github.com/vsocialtommy-del/tradingbot.git"
REPO_DIR = "/content/tradingbot" if IN_COLAB else "."

if IN_COLAB and not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned to {REPO_DIR}")
elif IN_COLAB:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    print(f"Pulled latest into {REPO_DIR}")

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# Imports.
import numpy as np
import pandas as pd

from bot.backtest import (
    BacktestConfig,
    BacktestEngine,
    BacktestResult,
    load_dukascopy_csv,
    generate_equity_curve,
    generate_drawdown_chart,
    generate_trade_scatter,
    generate_r_multiple_histogram,
    generate_hourly_heatmap,
    generate_skip_reasons_pie,
    generate_html_report,
)
print("bot.backtest imported.")


## Section 2 — Load data

The backtest expects an M5 OHLC DataFrame with `open / high / low /
close` columns and a tz-aware UTC `DatetimeIndex`. The
`load_dukascopy_csv` helper handles the standard Dukascopy export
format (and accepts ISO 8601 timestamps too, for non-Dukascopy
sources).

### Where to get real data

1. Visit <https://www.dukascopy.com/swiss/english/marketwatch/historical/>
2. Symbol: **XAU/USD**
3. Timeframe: **5 Min**
4. Date range: at least **1 year** (the more, the better)
5. Format: **CSV**, UTC timezone
6. Click **Download**

### Two ways to feed it in

* **Colab**: run the upload cell below (it pops a file picker)
* **Local**: drop the CSV next to this notebook and edit `CSV_PATH`

If you skip both, the notebook falls back to a synthetic dataset
generated below — usable to verify the pipeline works end-to-end,
but **not** representative of real market behaviour.


In [ ]:
# Synthetic XAUUSD M5 generator — fallback when no real CSV is supplied.
#
# Produces ~1000 bars (about 3 days of M5 with weekend gaps ignored)
# with seeded W/M patterns, BoS events, and realistic volatility.
# The strategy filters are strict; depending on params you may see
# zero or a handful of setups. That's fine for a smoke test.

def generate_synthetic_xauusd(n_bars: int = 1000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    # Base $2300 + small upward drift over the run.
    base = 2300.0
    drift = np.linspace(0, 25.0, n_bars)
    # Slow swing of ~$15 amplitude every ~50 bars (~4 hours).
    swing = 15.0 * np.sin(2 * np.pi * np.arange(n_bars) / 50)
    # Faster oscillation for finer structure.
    fast = 5.0 * np.sin(2 * np.pi * np.arange(n_bars) / 13)
    # Random walk noise — mean-reverting via cumulative scaling.
    noise = np.cumsum(rng.normal(scale=2.0, size=n_bars)) * 0.05
    closes = base + drift + swing + fast + noise
    opens = np.concatenate([[closes[0]], closes[:-1]])
    opens += rng.normal(scale=0.3, size=n_bars)
    bar_range = np.abs(rng.normal(scale=2.0, size=n_bars)) + 1.0
    highs = np.maximum(opens, closes) + bar_range / 2
    lows  = np.minimum(opens, closes) - bar_range / 2
    times = pd.date_range(
        "2026-01-01T00:00:00Z", periods=n_bars, freq="5min", tz="UTC",
    )
    return pd.DataFrame(
        {
            "open": opens, "high": highs, "low": lows,
            "close": closes, "volume": [100] * n_bars,
        },
        index=times,
    )

print("Synthetic data generator ready. Call generate_synthetic_xauusd() to produce a DataFrame.")


In [ ]:
# OPTIONAL: upload a Dukascopy CSV (Colab only).
# Skip this cell to use the synthetic data below.

CSV_PATH = None

if IN_COLAB:
    from google.colab import files
    print("Upload your XAUUSD M5 CSV (or click Cancel to use synthetic data):")
    try:
        uploaded = files.upload()
        if uploaded:
            CSV_PATH = f"/content/{next(iter(uploaded))}"
            print(f"Will load: {CSV_PATH}")
    except Exception as e:
        print(f"Upload skipped: {e}")
else:
    # Local users: edit this path to point at your CSV.
    # CSV_PATH = "./xauusd_m5.csv"
    pass


In [ ]:
# Load the data. Falls back to the synthetic generator if no CSV is set.
#
# Default = 200 synthetic bars (~17 hours of M5). Keeps the smoke-test
# under a couple of seconds even on Colab's slower CPU runtime — the
# strategy pipeline runs once per bar so wall-clock scales with bar
# count. Bump n_bars freely once you've verified the pipeline; the
# real backtest target is 1+ year of Dukascopy data anyway.
if CSV_PATH:
    df = load_dukascopy_csv(CSV_PATH)
    print(f"Loaded REAL data: {len(df)} bars from {df.index[0]} to {df.index[-1]}.")
else:
    df = generate_synthetic_xauusd(n_bars=200)
    print(f"Loaded SYNTHETIC data: {len(df)} bars from {df.index[0]} to {df.index[-1]}.")
    print("⚠ Replace with a real Dukascopy CSV for meaningful backtest results.")

print("\nFirst 3 bars:")
print(df.head(3))
print("\nLast 3 bars:")
print(df.tail(3))

In [ ]:
# Quick data quality check.
print(f"Bars:           {len(df):,}")
print(f"Date range:     {df.index[0]} → {df.index[-1]}")
print(f"Span:           {df.index[-1] - df.index[0]}")
print(f"Missing close:  {int(df['close'].isna().sum())}")
print(f"High < low:     {int((df['high'] < df['low']).sum())}")
print(f"Mean close:     ${df['close'].mean():.2f}")
print(f"Mean range:     ${(df['high'] - df['low']).mean():.2f}")


## Section 3 — Configure the backtest

`BacktestConfig` has sensible defaults that match the spec. The most
common ones to tune:

| Field | What it controls | Default |
|---|---|---|
| `starting_balance` | Fake money the bot starts with | $10,000 |
| `spread_points` | Vantage Gold raw spread (1 pt = $0.01) | 23 (= $0.23) |
| `slippage_points` | How much worse than expected fills are | 1.5 (= $0.015) |
| `commission_per_lot` | Vantage Raw account commission | $3.50 |
| `fixed_lot_size` | v1 sizing — 0.01 lots for everything | 0.01 |
| `max_simultaneous_setups` | Exposure cap from spec | 3 |
| `daily_loss_limit_pct` | Daily halt threshold | 10% |
| `sl_buffer_points` | Anti-stop-hunt SL buffer (price units, $17.50) | 17.5 |
| `tp1_method` | `"BOS_LEVEL"` or `"FIXED_DISTANCE"` | `"BOS_LEVEL"` |

**Units note:** `spread_points` and `slippage_points` use the broker
convention (1 pt = $0.01). Everything else (`sl_buffer_points`,
`layer_*_offset_points`, `tp1_fixed_distance_points`) is in **price
units** = dollars. This dual convention is inherited from the live
config; standardising would require a coordinated migration.


In [ ]:
# Default config — works for 99% of cases. Tweak below for experiments.
config = BacktestConfig(
    starting_balance=10_000.0,
    spread_points=23.0,
    slippage_points=1.5,
    commission_per_lot=3.50,
    fixed_lot_size=0.01,
    max_simultaneous_setups=3,
    daily_loss_limit_pct=10.0,
    sl_buffer_points=17.5,
    tp1_method="BOS_LEVEL",
    progress_log_every_bars=50,  # heartbeat every 50 bars — keeps Colab UI responsive
)

In [ ]:
# Print the resolved config (helpful for the report later).
from dataclasses import asdict, fields, is_dataclass
def _config_table(cfg) -> None:
    for f in fields(cfg):
        v = getattr(cfg, f.name)
        if is_dataclass(v):
            print(f"{f.name}:")
            for sf in fields(v):
                print(f"  {sf.name:30s} = {getattr(v, sf.name)!r}")
        else:
            print(f"{f.name:30s} = {v!r}")

_config_table(config)


## Section 4 — Run the backtest

Approximate runtimes on a Colab CPU runtime:

| Bars | ≈ Wall clock |
|---|---|
| 1,000 (synthetic) | < 1 s |
| 50,000 (~6 months M5) | 10-30 s |
| 200,000 (~2 years M5) | 1-3 min |

The bottleneck is the strategy pipeline, run once per M5 bar. Progress
logs every `progress_log_every_bars` bars (set to 200 above).


In [ ]:
import time
t0 = time.time()
result = BacktestEngine(config).run(df)
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s — {result.bars_processed} bars processed.")


In [ ]:
# Headline metrics.
m = result.metrics
print(f"=== Equity ===")
print(f"  Starting:        ${m.equity.starting_balance:,.2f}")
print(f"  Ending:          ${m.equity.ending_balance:,.2f}")
print(f"  Net P&L:         ${m.trades.net_pnl:+,.2f}")
print(f"  Total return:    {m.equity.total_return_pct:+.2f}%")
print(f"  Max drawdown:    {m.equity.max_drawdown_pct:.2f}% (${m.equity.max_drawdown_dollars:,.2f})")
print(f"  Sharpe (ann):    {m.equity.sharpe_ratio:.2f}")
print()
print(f"=== Trades ===")
print(f"  Total:           {m.trades.total}")
print(f"  Win rate:        {m.trades.win_rate * 100:.1f}%")
pf = '∞' if m.trades.profit_factor == float('inf') else f'{m.trades.profit_factor:.2f}'
print(f"  Profit factor:   {pf}")
print(f"  Expectancy:      ${m.trades.expectancy:+.2f} / trade")
print(f"  Avg R-multiple:  {m.trades.avg_r_multiple:+.2f}R")
print()
print(f"=== Setups ===")
print(f"  Detected:        {m.setups.detected}")
print(f"  Taken:           {m.setups.taken}")
print(f"  TP1 hit rate:    {m.setups.tp1_hit_rate * 100:.1f}%")
print(f"  SL stop rate:    {m.setups.sl_stop_rate * 100:.1f}%")
if m.setups.skip_reasons:
    print(f"  Skip reasons:    {dict(m.setups.skip_reasons)}")


## Section 5 — Visualise

Six charts, each annotated with what to look for. They render inline
in Colab and Jupyter. (For an offline shareable artefact, see the
HTML report in Section 6.)


### Equity curve

Account balance over time. TP1 hits are blue triangles, SL hits are
orange triangles. The annotation marks the deepest drawdown.

**Healthy run:** smoothly rising line, drawdowns shallow and short.
**Suspect:** big drawdowns, or a flat line followed by a cliff
(martingale / averaging-down behaviour).


In [ ]:
generate_equity_curve(result).show()


### Drawdown

Underwater chart: % from running peak.

**Healthy:** drawdowns recover within a reasonable window.
**Suspect:** extended periods underwater, or single deep notches.


In [ ]:
generate_drawdown_chart(result).show()


### Trades on price

Entries on the underlying close. Winners are blue triangles, losers orange.

**Healthy:** entries cluster near supports/resistances; winners and
losers are scattered through the run.
**Suspect:** all winners early, all losers late (regime change), or
all entries clustered at one price (overfitting).


In [ ]:
generate_trade_scatter(result, df=df).show()


### R-multiple distribution

Histogram of trade outcomes in R-multiples (trade P&L ÷ initial risk).
Reference vlines at -1R, 0R, +1R.

**Healthy:** mode near 0R, fat right tail (winners can run beyond
+1R via TP-then-runner), thin left tail capped at -1R.
**Suspect:** mode at +1R only (no runners working), or losers
beyond -1R (slippage / gap risk).


In [ ]:
generate_r_multiple_histogram(result).show()


### Hourly heatmap

Day of week × hour of day, coloured by trade count. Cells with no
trades are blank.

**Healthy:** activity concentrated in liquid sessions (London 7-12 UTC,
NY 13-18 UTC), with minimal Asian-session activity.
**Suspect:** heavy weekend/Friday-evening activity (illiquid).


In [ ]:
generate_hourly_heatmap(result).show()


### Skip reasons

Why detected zones didn't become setups. Helps tune filter parameters.

**Common categories:**
* `exposure_cap` — bot already had `max_simultaneous_setups` open;
  zones got refused. Increasing the cap raises risk.
* `sl_too_close` / `sl_too_far` — SL distance outside
  `[min_sl_distance_points, max_sl_distance_points]`. Adjust the
  bounds OR look at the zones being rejected.
* `sl_calc_failed` — should be 0 in normal operation; flag a bug.


In [ ]:
generate_skip_reasons_pie(result).show()


## Section 6 — Generate HTML report

Combines all six charts plus a metrics summary into one self-contained
HTML file (~3 MB; Plotly JS embedded inline so it works offline).
Useful for sharing with non-coders or archiving runs.


In [ ]:
report_path = generate_html_report(
    result,
    output_path="/content/backtest_report.html" if IN_COLAB else "./backtest_report.html",
    df=df,
    title=f"Backtest — {df.index[0].date()} to {df.index[-1].date()}",
)
print(f"Wrote: {report_path}")


In [ ]:
# Colab: download the file. Local: just open it in your browser.
if IN_COLAB:
    from google.colab import files
    files.download(report_path)
else:
    print(f"Open in browser: file://{report_path}")


## Section 7 — Iteration

### How to tune parameters

1. Open Section 3, change one or two fields on `BacktestConfig`.
2. Re-run Sections 4-6.
3. Compare metrics; archive the report (rename `backtest_report.html`).

### What to look for

**Strong run:**
* Profit factor > 1.5
* Win rate > 40% combined with avg R-multiple > 0.5R
* Max drawdown < 15%
* Sharpe > 1.0
* TP1 hit rate ≥ 50% (matches spec target)

**Suspect run:**
* PF < 1.0 across multiple data ranges → strategy logic flaw
* Big winning streaks followed by big losses → regime sensitivity
* Low setup count (e.g. < 1 / week) → filters too strict
* High setup count with poor PF → filters too loose

### Important caveats

* **No news filter** in v1 backtest — Finnhub historical data isn't
  available in our pipeline. Real performance may differ on NFP,
  CPI, FOMC days.
* **Tick approximation** — the OHLC walk method is the industry
  fallback when real ticks aren't available, but it's still an
  approximation. Same-bar SL+TP conflict resolves SL-wins
  (pessimistic) by default.
* **Synthetic data** — the seeded data in this notebook is for
  smoke-testing the pipeline only. Use real Dukascopy data for
  any tuning or evaluation.

### Example: looser exposure cap


In [ ]:
# Example: bump max_simultaneous_setups from 3 to 5 to see the effect.
config_loose = BacktestConfig(
    starting_balance=config.starting_balance,
    max_simultaneous_setups=5,        # changed from 3
    daily_loss_limit_pct=config.daily_loss_limit_pct,
    progress_log_every_bars=50,
)
result_loose = BacktestEngine(config_loose).run(df)
print(f"Setups taken — default 3-cap: {result.setups_taken}")
print(f"Setups taken — looser 5-cap: {result_loose.setups_taken}")
print(f"P&L delta:                    ${result_loose.metrics.trades.net_pnl - result.metrics.trades.net_pnl:+.2f}")

---

That's the full backtest workflow. Next steps:

1. **Replace the synthetic data with real Dukascopy XAUUSD M5** (Section 2).
2. **Run a baseline backtest** with default config and archive the report.
3. **Tune parameters** one at a time; compare reports side-by-side.
4. **Phase F**: deploy the live bot to a VPS for demo trading and
   compare live results against the backtest baseline.
